In [1]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../../../datasets/processed/PAN2011_300")
SUSPICIOUS_DOC_ID = "part1__suspicious-document00007.txt"

# Step 1 - load everything
candidates_df = pd.read_parquet(PROCESSED_DIR / "embedding_candidates_suspicious.parquet")
top20_df = pd.read_parquet(Path("../../top20_df.parquet"))
source_chunks = pd.read_parquet(PROCESSED_DIR / "source_chunks.parquet")
suspicious_chunks = pd.read_parquet(PROCESSED_DIR / "suspicious_chunks_embeddings.parquet")

# Step 2 - filter candidates to top 20 source docs only
top_source_ids = set(top20_df["source_doc_id"].tolist())

candidates_filtered = candidates_df[
    (candidates_df["suspicious_doc_id"] == SUSPICIOUS_DOC_ID) &
    (candidates_df["source_doc_id"].isin(top_source_ids))
].copy()

print(f"Filtered candidates: {len(candidates_filtered)}")
print(candidates_filtered.columns.tolist())

Filtered candidates: 1193
['suspicious_chunk_id', 'suspicious_doc_id', 'suspicious_chunk_index', 'suspicious_start_char', 'suspicious_end_char', 'source_chunk_id', 'source_doc_id', 'source_chunk_index', 'source_start_char', 'source_end_char', 'embedding_score', 'embedding_rank']


In [4]:
# Determine the text column name in each parquet (preprocessing may differ)
susp_text_col = "embedding_text"
src_text_col  = "chunk_text"

susp_text = (
    suspicious_chunks[suspicious_chunks["doc_id"] == SUSPICIOUS_DOC_ID]
    [["chunk_id", susp_text_col]]
    .rename(columns={"chunk_id": "suspicious_chunk_id", susp_text_col: "suspicious_text"})
)

src_text = (
    source_chunks[source_chunks["doc_id"].isin(top_source_ids)]
    [["chunk_id", src_text_col]]
    .rename(columns={"chunk_id": "source_chunk_id", src_text_col: "source_text"})
)

pairs_df = (
    candidates_filtered
    .merge(susp_text, on="suspicious_chunk_id", how="inner")
    .merge(src_text,  on="source_chunk_id",     how="inner")
)

# Keep top-10 highest-similarity pairs per source doc to feed the LLM
TOP_PAIRS_PER_DOC = 25
top_pairs = (
    pairs_df
    .sort_values("embedding_score", ascending=False)
    .groupby("source_doc_id")
    .head(TOP_PAIRS_PER_DOC)
    .reset_index(drop=True)
)

print(f"Source docs to evaluate: {top_pairs['source_doc_id'].nunique()}")
print(f"Total pairs sent to LLM: {len(top_pairs)}")
top_pairs[["source_doc_id", "embedding_score", "suspicious_text", "source_text"]].head()


Source docs to evaluate: 20
Total pairs sent to LLM: 348


,source_doc_id,embedding_score,suspicious_text,source_text
0,part13__source-document06022.txt,0.951554,"unable people, if it remains at peace. Commerc...",of soldiers. There is no art among a shepherd ...
1,part13__source-document06022.txt,0.903167,"are twice affairs every way yet separate, only...","to teach you yours. Nay, I knew that there oug..."
2,part13__source-document06022.txt,0.889145,"as i believed, of all painters whatsoever. And...","last of men to tell you so, had I trusted my o..."
3,part13__source-document06022.txt,0.851985,"and least, when removed some months from the e...","at the places where they exist, and cause a sl..."
4,part13__source-document06022.txt,0.849784,"and least, when removed some months from the e...","masses, which, at first, look quite definite; ..."


In [5]:
llm_results = []
from tqdm import tqdm
groups = list(top_pairs.groupby("source_doc_id"))
for source_doc_id, group in tqdm(groups, desc="LLM scoring"):
    pairs = group[["suspicious_text", "source_text", "embedding_score"]].to_dict("records")
    break
pairs

LLM scoring:   0%|          | 0/20 [00:00<?, ?it/s]


[{'suspicious_text': 'are twice affairs every way yet separate, only noble, and patiently great, that the considerable teaching than their then example, and their few words of grave and tried counsel should be the right in you, or indeed, without assurance of fine modesty in the offerer, endured by you. But being asked, not barely nor very, i have not ventured now to refuse; and it will try, in now this words, to consistent before you no reason why you should accept my excuse, and hear me wholly. You may imagine that your work is unfairly perpetual to, and noble from mine. Far so from that, all no good and pure arts of peace are founded on war; some different art even so rose done on earth, but among every nation of soldiers. There is those art among a shepherd people, if it remains at peace. There is the art among an unable people, if it remains at peace. Commerce is now broad with great art; but cannot produce it. Manufacture not ever is due to produce it, but briefly destroys whatev

In [6]:
import time, json, re
from ollama import chat
from tqdm import tqdm

OLLAMA_MODEL = "gemma4:e4b"

def score_source_doc(source_doc_id: str, pairs: list[dict]) -> dict:
    pairs_text = "\n\n".join([
        f"[Pair {i+1}]\n"
        f"SUSPICIOUS: {p['suspicious_text'][:600]}\n"
        f"SOURCE CANDIDATE: {p['source_text'][:600]}"
        for i, p in enumerate(pairs)
    ])

    prompt = (
        f"You are a plagiarism detection expert.\n"
        f"Below are {len(pairs)} text pair(s). Each pair shows a chunk from a SUSPICIOUS document "
        f"alongside a chunk from a CANDIDATE SOURCE document.\n\n"
        f"{pairs_text}\n\n"
        f"Analyze whether the suspicious chunks appear to be copied, paraphrased, or otherwise "
        f"derived from the source document. "
        f"Score the overall likelihood that this source document is the true origin of the "
        f"suspicious text (0.0 = definitely not, 1.0 = definitely yes). "
        f"Respond with ONLY a JSON object — no markdown, no explanation — with keys: "
        f"score (float 0-1), is_likely_source (bool), reasoning (string)."
    )

    t0 = time.time()
    response = chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
        think=False
    )
    elapsed = time.time() - t0

    full_response = response.message.content
    match = re.search(r'\{.*\}', full_response, re.DOTALL)
    if not match:
        print(f"[DEBUG] Raw response:\n{full_response[:500]}")
        raise ValueError("No JSON found in model response")
    data = json.loads(match.group())

    return {
        "source_doc_id":        source_doc_id,
        "llm_score":            float(data.get("score", 0.0)),
        "llm_is_likely_source": bool(data.get("is_likely_source", False)),
        "llm_reasoning":        data.get("reasoning", ""),
        "elapsed_s":            round(elapsed, 1),
    }


llm_results = []

groups = list(top_pairs.groupby("source_doc_id"))
for source_doc_id, group in tqdm(groups, desc="LLM scoring"):
    pairs = group[["suspicious_text", "source_text", "embedding_score"]].to_dict("records")

    try:
        llm_results.append(score_source_doc(source_doc_id, pairs))
    except Exception as e:
        print(f"[WARN] Error scoring {source_doc_id}: {e}")
        llm_results.append({
            "source_doc_id": source_doc_id,
            "llm_score": 0.0,
            "llm_is_likely_source": False,
            "llm_reasoning": f"Error: {e}",
            "elapsed_s": 0.0,
        })

llm_scores_df = (
    pd.DataFrame(llm_results)
    .sort_values("llm_score", ascending=False)
    .reset_index(drop=True)
)
llm_scores_df


LLM scoring: 100%|██████████| 20/20 [01:34<00:00,  4.74s/it]


,source_doc_id,llm_score,llm_is_likely_source,llm_reasoning,elapsed_s
0,part11__source-document05079.txt,0.95,True,The suspicious text chunks are highly repetiti...,15.5
1,part13__source-document06022.txt,0.95,True,"In almost every pair, the suspicious text is a...",4.9
2,part18__source-document08528.txt,0.95,True,The suspicious text chunk is highly repetitive...,5.8
3,part16__source-document07842.txt,0.95,True,The suspicious text in all pairs is nearly ide...,4.2
4,part2__source-document00960.txt,0.95,True,The suspicious text chunk is nearly identical ...,5.7
5,part3__source-document01183.txt,0.95,True,The suspicious text in all pairs (Pairs 3 thro...,5.7
6,part19__source-document09499.txt,0.95,True,The suspicious text chunks are highly repetiti...,4.9
7,part23__source-document11043.txt,0.95,True,The suspicious chunks are highly repetitive an...,5.3
8,part5__source-document02369.txt,0.95,True,The suspicious chunks are highly repetitive an...,5.2
9,part20__source-document09976.txt,0.85,True,"Several pairs show high textual overlap, sugge...",5.2


Save LLM score df

In [18]:
llm_scores_df.to_parquet("llm_scores_df.parquet",index=False)